# Laya-1.2-speedup — Train first, validate, then publish the model

**V4.6 derived model.** Based on the successful Tesla T4 marker-aware score-only experiment: a 64-context-token compressed decision head, 100% agreement on the 200-decision V4.6 audit and 1.203× measured speed. Those are *historical experimental results*, **not** promises about this run.

This Colab does **all steps in the correct order**:

1. Loads the original pretrained Laya typed-decisions checkpoint and case-disjoint TRAIN / PROBE / AUDIT splits.
2. **Trains** the 64-token decision head with teacher distillation, keeping the 28-layer ModernBERT backbone and its FFNs/QKV/RoPE **frozen**.
3. Uses PROBE-only comparisons to choose a trained head **only if fidelity and speed remain acceptable**; otherwise retains the untrained pretrained-head control.
4. Optionally compiles the selected head and reruns an untouched 200-decision AUDIT with interleaved wall and CUDA timing.
5. Saves `head.safetensors`, `modeling_laya_speedup.py` (loader), `config.json`, metrics and a model card. **The heavyweight original pretrained backbone is referenced, not republished.**
6. After *your* write-authorized Hugging Face login, creates `vtava/Laya-1.2-speedup` as a **model** repository (private by default) and uploads it **only after validation**.

**Important scope:** Optimized inference produces option scores only, not auxiliary action logits. “Laya-1.2-speedup” is a research run name, not a universal 1.2× claim. A failed validation never triggers a “successful model” upload.


In [ ]:
#@title 1. Dependencies and GPU
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "In Colab select Runtime > Change runtime type > GPU."
print("Torch",torch.__version__,"CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Reproducible source model and experiment configuration
from pathlib import Path
import math, json, random, time, copy, gc, statistics
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (QTYPES, build_checkpoint_model,
    build_sequence, collate_items, normalize_question)

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2046
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
PROBE_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
TARGET_BUDGET=64 # fixed from V4.6 PROBE-only research selection
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.99 #@param {type:"number"}
MAX_GOLD_DROP_PP=1.5 #@param {type:"number"}
MIN_SPEEDUP=1.05 #@param {type:"number"}
ENABLE_HEAD_COMPILE=True #@param {type:"boolean"}
TRAIN_STEPS=120 #@param {type:"integer"}
TRAIN_CHECK_EVERY=20 #@param {type:"integer"}
TRAIN_LR=0.00003 #@param {type:"number"}
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}
OUT=Path("/content/Laya_1p2_speedup")
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/Laya_1p2_speedup")
OUT.mkdir(parents=True,exist_ok=True)
assert 1<=BATCH<=8 and BENCH_REPEATS>=5 and 0<MIN_TEACHER_AGREEMENT<=1
assert TARGET_BUDGET==64 and TRAIN_STEPS>=0 and TRAIN_CHECK_EVERY>=1
assert MAX_GOLD_DROP_PP>=0 and MIN_SPEEDUP>=1.0
random.seed(SEED); np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(SOURCE_MODEL, allow_patterns=[
    "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
assert teacher.head is not None, "V4.6 requires a pretrained decision Transformer head"
assert MAX_LEN <= int(source_cfg.get("max_len",MAX_LEN))
print("Original model parameters:",sum(p.numel() for p in teacher.parameters()))
print("Decision head layers:",len(teacher.head.layers),"AMP dtype",amp_dtype)


In [ ]:
#@title 3. Case-disjoint TRAIN / PROBE / AUDIT data + batch utilities
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

assert len(probe_items)>=PROBE_DECISIONS and len(audit_items)>=AUDIT_DECISIONS
assert {x["case_id"] for x in probe_items}.isdisjoint({x["case_id"] for x in audit_items})
print("TRAIN/PROBE/AUDIT items:",len(train_items),len(probe_items),len(audit_items))


In [ ]:
#@title 4. Architecture: native ModernBERT + marker-preserving decision head
# This model shares the original FROZEN pretrained weights. NO duplicated encoder,
# no CeNN branch, no learned new parameters, no training.
class MarkerAwareScores(nn.Module):
    def __init__(self, pretrained, budget=0):
        super().__init__()
        self.encoder=pretrained.encoder
        self.type_emb=pretrained.type_emb
        self.head=pretrained.head
        self.scorer=pretrained.scorer
        self.budget=int(budget)
        assert self.budget>=0

    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):
        # Keep CLS for the original typed-decision semantics; preserve *all*
        # option marker positions, then sample a fixed number of context tokens.
        # Duplicate markers and padded context are masked rather than attended.
        b,t,d=h.shape
        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)
        lm_count=min(self.budget,max(t-1,1))
        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()
        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)
        marker_ok=marker_mask.bool()
        lm_ok=attention_mask.bool().gather(1,lm)
        lm_ok=lm_ok & (lm!=0)
        # Markers may be padded to 0; their mask distinguishes real positions.
        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]
        lm_ok=lm_ok & ~duplicate.any(-1)
        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)
        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),
                        marker_ok,lm_ok),dim=1)
        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))
        return gathered,keep

    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):
        h=self.encoder(input_ids=input_ids,
                       attention_mask=attention_mask).last_hidden_state
        h=h+self.type_emb(qtype)[:,None,:]
        if self.budget:
            compact,keep=self._select_head_tokens(h,attention_mask,
                                                    marker_pos,marker_mask)
            transformed=self.head(compact,src_key_padding_mask=~keep)
            marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]
        else:
            transformed=self.head(h,src_key_padding_mask=~attention_mask.bool())
            marker_hidden=transformed.gather(1,marker_pos.clamp_min(0)[:,:,None]
                                             .expand(-1,-1,h.shape[-1]))
        logits=self.scorer(marker_hidden).squeeze(-1).float()
        logits=logits.masked_fill(~marker_mask,-1e4)
        # The original action logits are deliberately NOT computed here:
        # this variant serves option-score decisions only.
        return logits,None

exact_score_model=MarkerAwareScores(teacher,budget=0).eval()
assert all(not p.requires_grad for p in exact_score_model.parameters())
print("Ready: budget=0 reproduces the original option logits, omitting only act_logits")


In [ ]:
#@title 5. Strict quality tests and A/B speed measurement
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=0.003)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_SPEEDUP and
            s["min_wall_speedup"]>=MIN_SPEEDUP)

PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
exact_q=quality(exact_score_model,probe_items,PROBE_REF)
print("Exact score-only PROBE:",exact_q)
assert exact_q["teacher_agreement"]==1.0 and abs(exact_q["mean_js"])<1e-7, (
    "Score-only implementation drifted from native pretrained teacher")
# Mandatory exact-output smoke test on a real, padded PROBE batch.
with torch.inference_mode(), torch.autocast("cuda",dtype=amp_dtype):
    _smoke=batch(probe_items,0,BATCH)
    _original_scores,_=teacher(**_smoke)
    _fast_scores,_=exact_score_model(**_smoke)
    _max_diff=float((_original_scores.float()-_fast_scores.float()).abs().max())
    assert torch.allclose(_original_scores.float(),_fast_scores.float(),
                          rtol=1e-5,atol=1e-5),_max_diff
print("Exact option-logit parity checked; max absolute error:",_max_diff)
print("Native benchmark shapes:",[(int(b["input_ids"].shape[0]),
                                  int(b["input_ids"].shape[1])) for b in BENCH_BATCHES])


In [ ]:
#@title 6. Train the 64-context-token head (no encoder or FFN finetuning)
# V4.6 found 64 tokens best on the PROBE sweep; do NOT use AUDIT to choose budget.
# We always execute distillation training before trying to publish a new model.
import copy
import numpy as np

baseline_model=MarkerAwareScores(teacher,budget=TARGET_BUDGET).eval()
baseline_probe=quality(baseline_model,probe_items,PROBE_REF)
baseline_speed=benchmark_paired(teacher,baseline_model)
assert valid_quality(baseline_probe) and valid_speed(baseline_speed), (
    "V4.6 architecture did not reproduce acceptable PROBE quality and speed; "
    "do not train/upload under the previously reported acceleration name.",
    baseline_probe,baseline_speed)
print("BEFORE TRAIN:",baseline_probe)
print("BEFORE TRAIN SPEED:",baseline_speed["median_gpu_speedup"])

student=MarkerAwareScores(teacher,budget=TARGET_BUDGET).eval()
student.head=copy.deepcopy(teacher.head).to(device)
student.encoder.eval().requires_grad_(False)
student.type_emb.eval().requires_grad_(False)
student.scorer.eval().requires_grad_(False)
student.head.train().requires_grad_(True)
optimizer=torch.optim.AdamW(student.head.parameters(),lr=TRAIN_LR,weight_decay=0.)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)

def head_snapshot(module):
    return {key:value.detach().cpu().contiguous().clone()
            for key,value in module.head.state_dict().items()}

selection_records=[]
best_step=0
best_head_state=head_snapshot(baseline_model)
best_probe=baseline_probe
best_speed=baseline_speed

for step in range(1,TRAIN_STEPS+1):
    b=batch(train_items,((step-1)*BATCH)%len(train_items),BATCH)
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**b)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        student_logits,_=student(**b)
        temp=2.0
        target=F.softmax(teacher_logits.float()/temp,dim=-1)
        output_log=F.log_softmax(student_logits.float()/temp,dim=-1)
        loss=F.kl_div(output_log,target,reduction="batchmean")*(temp**2)
    if not torch.isfinite(loss):
        raise RuntimeError("Nonfinite training loss; no checkpoint selected.")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(student.head.parameters(),1.0)
    scaler.step(optimizer)
    scaler.update()

    if step%TRAIN_CHECK_EVERY==0 or step==TRAIN_STEPS:
        student.head.eval()
        candidate_q=quality(student,probe_items,PROBE_REF)
        candidate_speed=(benchmark_paired(teacher,student)
                         if valid_quality(candidate_q) else None)
        feasible=(candidate_speed is not None and valid_speed(candidate_speed))
        # Training is selected when it improves teacher distribution fidelity,
        # without a material >3% median latency regression vs best accepted.
        better=bool(feasible and
            candidate_q["mean_js"]+1e-9 < best_probe["mean_js"] and
            candidate_speed["median_gpu_speedup"]>=
                    0.97*best_speed["median_gpu_speedup"])
        # Alternatively accept a meaningfully faster trained head with
        # essentially unchanged teacher-distribution fidelity.
        faster=bool(feasible and
            candidate_speed["median_gpu_speedup"]>=
                    1.01*best_speed["median_gpu_speedup"] and
            candidate_q["mean_js"]<=best_probe["mean_js"]+1e-6)
        if better or faster:
            best_head_state=head_snapshot(student)
            best_step=step
            best_probe=candidate_q
            best_speed=candidate_speed
        rec=dict(step=step,train_loss=float(loss.detach().cpu()),
            teacher_agreement=candidate_q["teacher_agreement"],
            gold_accuracy=candidate_q["gold_accuracy"],
            gold_drop=candidate_q["gold_drop"],
            mean_js=candidate_q["mean_js"],
            median_speedup=None if candidate_speed is None else candidate_speed["median_gpu_speedup"],
            min_speedup=None if candidate_speed is None else candidate_speed["min_gpu_speedup"],
            selected=bool(better or faster))
        selection_records.append(rec)
        (OUT/"v46_training_progress.json").write_text(json.dumps(selection_records,indent=2))
        print("TRAIN",rec)
        student.head.train()
student.head.eval().requires_grad_(False)

# Rehydrate the frozen chosen head, without retaining optimizer or teacher ties.
chosen_eager=MarkerAwareScores(teacher,budget=TARGET_BUDGET).eval()
chosen_eager.head=copy.deepcopy(teacher.head).to(device)
chosen_eager.head.load_state_dict(best_head_state,strict=True)
chosen_eager.head.eval().requires_grad_(False)
selected_head_kind="trained" if best_step>0 else "pretrained_unmodified"
print("PROBE-selected head:",selected_head_kind,"training step:",best_step)
if TRAIN_STEPS>0 and best_step==0:
    print("Training ran, but did NOT improve the quality/speed gates; unchanged "
          "head retained. Publication metadata will say 'pretrained_unmodified'.")


In [ ]:
#@title 7. PROBE-only runtime selection: optional head torch.compile
# Compiled weights are the exact same chosen head weights, not new learned weights.
from torch import nn
runtime_records=[]
runtime_candidates=[("eager",chosen_eager)]
if ENABLE_HEAD_COMPILE:
    try:
        compiled=MarkerAwareScores(teacher,budget=TARGET_BUDGET).eval()
        compiled.head=torch.compile(chosen_eager.head,mode="reduce-overhead",dynamic=False)
        with torch.inference_mode():
            for b in BENCH_BATCHES:
                for _ in range(4):
                    with torch.autocast("cuda",dtype=amp_dtype):
                        compiled(**b)
            torch.cuda.synchronize()
        runtime_candidates.append(("compile_head",compiled))
    except Exception as ex:
        print("torch.compile unavailable:",repr(ex))
        torch.cuda.empty_cache()

runtime_selection=None
runtime_best_gpu=0.
for label,candidate in runtime_candidates:
    p=quality(candidate,probe_items,PROBE_REF)
    perf=benchmark_paired(teacher,candidate) if valid_quality(p) else None
    passed=(perf is not None and valid_speed(perf))
    score=perf["median_gpu_speedup"] if perf else 0.
    runtime_records.append(dict(name=label,quality=p,speed=perf,
                                passed=passed))
    # Only select compiled over eager if a measurable repeatable win.
    if passed and (runtime_selection is None or
                   score>=runtime_best_gpu*1.003):
        runtime_selection=(label,candidate)
        runtime_best_gpu=score
    print(label,"fidelity",p["teacher_agreement"],"speedup",score,
          "passed",passed)

assert runtime_selection is not None, (
    "No speed/fidelity-valid runtime after training; publication is blocked")
runtime_name,chosen_runtime=runtime_selection
(OUT/"v46_runtime_probe.json").write_text(json.dumps(runtime_records,indent=2))
print("FROZEN runtime",runtime_name,"head kind",selected_head_kind,
      "training step",best_step)


In [ ]:
#@title 8. Untouched AUDIT (no model selection using AUDIT)
audit_reference=get_reference(audit_items,AUDIT_DECISIONS)
assert {r["case"] for r in audit_reference}.isdisjoint(
       {r["case"] for r in PROBE_REF})
final_quality=quality(chosen_runtime,audit_items,audit_reference)
final_speed=benchmark_paired(teacher,chosen_runtime,repeats=max(15,BENCH_REPEATS))
gates={
    "teacher_agreement":final_quality["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
    "gold_drop":final_quality["gold_drop"]<=MAX_GOLD_DROP_PP/100,
    "gpu_speedup":final_speed["min_gpu_speedup"]>=MIN_SPEEDUP,
    "wall_speedup":final_speed["min_wall_speedup"]>=MIN_SPEEDUP,
}
gates["all_pass"]=all(gates.values())
report=dict(
    version="V4.6-Laya-1.2-speedup",
    model_type="pretrained frozen ModernBERT + 64-token option-scoring head",
    source_model=SOURCE_MODEL,head_kind=selected_head_kind,
    training_steps_executed=int(TRAIN_STEPS),selected_training_step=best_step,
    selected_runtime=runtime_name,selected_context_budget=TARGET_BUDGET,
    output_scope="option-score logits only; auxiliary action head omitted",
    device=torch.cuda.get_device_name(0),pytorch=torch.__version__,
    selection_probe=dict(baseline=baseline_probe,
                         baseline_speed=baseline_speed,
                         training=selection_records,
                         runtime=runtime_records),
    final_audit=final_quality,final_paired_cuda=final_speed,gates=gates)
report_file=OUT/"v46_model_final_audit.json"
report_file.write_text(json.dumps(report,indent=2))
print("AUDIT",final_quality)
print("GPU SPEED",final_speed["median_gpu_speedup"],"x")
print("WALL SPEED",final_speed["median_wall_speedup"],"x")
print("VALIDATION GATES",gates)
if not gates["all_pass"]:
    print("STOP: this version is NOT verified faster and accurate. "
          "Do not publish under the speedup model name.")


In [ ]:
#@title 9. Export a reproducible head checkpoint and model loader
from safetensors.torch import save_file,load_file
import hashlib
import textwrap, zipfile
artifacts=OUT/"model_release"
artifacts.mkdir(exist_ok=True,parents=True)
head_file=artifacts/"head.safetensors"
save_file({k:v.detach().cpu().contiguous() for k,v in best_head_state.items()},
          str(head_file))
loader_code=r"""# Laya 1.2 speedup — frozen base + saved optimized decision head
# Install dependencies from requirements.txt. The 421M-param base is downloaded
# from the upstream source repository; only head.safetensors lives in this repo.
import json
from pathlib import Path
import torch
from torch import nn
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download, snapshot_download
from safetensors.torch import load_file
from tinycenn_lm.standalone_decision import build_checkpoint_model

class MarkerAwareScores(nn.Module):
    def __init__(self, pretrained, budget=64):
        super().__init__()
        self.encoder=pretrained.encoder
        self.type_emb=pretrained.type_emb
        self.head=pretrained.head
        self.scorer=pretrained.scorer
        self.budget=int(budget)
        assert self.budget>0

    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):
        b,t,d=h.shape
        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)
        lm_count=min(self.budget,max(t-1,1))
        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()
        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)
        marker_ok=marker_mask.bool()
        lm_ok=attention_mask.bool().gather(1,lm)
        lm_ok=lm_ok & (lm!=0)
        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]
        lm_ok=lm_ok & ~duplicate.any(-1)
        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)
        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),
                        marker_ok,lm_ok),dim=1)
        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))
        return gathered,keep

    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):
        h=self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        h=h+self.type_emb(qtype)[:,None,:]
        compact,keep=self._select_head_tokens(h,attention_mask,marker_pos,marker_mask)
        transformed=self.head(compact,src_key_padding_mask=~keep)
        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]
        logits=self.scorer(marker_hidden).squeeze(-1).float()
        logits=logits.masked_fill(~marker_mask,-1e4)
        return logits,None

def load_laya_speedup(repo_id="vtava/Laya-1.2-speedup", *,
                      device="cuda",compile_head=False):
    # torch.compile is optional, not serialized in the state dict.
    cfg=json.loads(Path(hf_hub_download(repo_id,filename="config.json")).read_text())
    src=Path(snapshot_download(cfg["source_model"],allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
    original,_=build_checkpoint_model(src)
    original=original.to(device).eval().requires_grad_(False)
    model=MarkerAwareScores(original,budget=int(cfg["head_context_budget"]))
    weights=load_file(hf_hub_download(repo_id,filename="head.safetensors"),
                      device="cpu")
    model.head.load_state_dict(weights,strict=True)
    model=model.to(device).eval().requires_grad_(False)
    model.encoder.config.reference_compile=False
    if compile_head:
        model.head=torch.compile(model.head,mode="reduce-overhead",dynamic=False)
    tokenizer=AutoTokenizer.from_pretrained(str(src/"tokenizer"))
    return model,tokenizer
"""
(artifacts/"modeling_laya_speedup.py").write_text(loader_code,encoding="utf-8")
cfg=dict(model_type="laya-v46-marker-head",source_model=SOURCE_MODEL,
         head_context_budget=TARGET_BUDGET,head_file="head.safetensors",
         source_commit="github:vtavakkoli/TinyCeNN-LM@main",
         architecture="original 28-layer ModernBERT + marker-aware 64-token post-encoder head",
         selected_weights=selected_head_kind,selected_training_step=best_step,
         runtime=runtime_name,output_scope="option-scores only",
         auxiliary_act_head_returned=False,
         validation_device=torch.cuda.get_device_name(0),
         audit_fidelity=final_quality,audit_speed=final_speed,
         all_gates_passed=bool(gates["all_pass"]))
(artifacts/"config.json").write_text(json.dumps(cfg,indent=2),encoding="utf-8")
(artifacts/"audit_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
(artifacts/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nhuggingface_hub>=0.25,<1\n"
    "safetensors>=0.4\n"
    "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main\n",
    encoding="utf-8")
card=f"""---
base_model: {SOURCE_MODEL}
library_name: pytorch
tags:
  - modernbert
  - inference-optimization
  - distillation
  - research
  - marker-aware
---

# Laya-1.2-speedup — V4.6 experimental option-scoring model

This is a **derived model** for the option-scoring part of
[{SOURCE_MODEL}](https://huggingface.co/{SOURCE_MODEL}).
The original frozen 28-layer ModernBERT encoder, FFNs, Q/K/V and RoPE
are downloaded from upstream. The only redistributed weights are the
post-encoder decision Transformer head in \`head.safetensors\`.

Training steps executed: **{TRAIN_STEPS}**. Selected head weights:
**{selected_head_kind}** (selected checkpoint step: **{best_step}**).
If step is 0, training was attempted but the original pretrained head
won the PROBE validation; do **not** describe the published head as finetuned.

## Independent held-out V4.6 AUDIT ({AUDIT_DECISIONS} option decisions)

- Teacher agreement: **{final_quality["teacher_agreement"]:.4%}**
- Gold accuracy: **{final_quality["gold_accuracy"]:.4%}**
- Teacher gold accuracy: **{final_quality["teacher_gold_accuracy"]:.4%}**
- Gold accuracy drop: **{final_quality["gold_drop"]:.4%}**
- Mean JS: **{final_quality["mean_js"]:.8f}**
- Paired T4 GPU median speed ratio: **{final_speed["median_gpu_speedup"]:.4f}x**
- Paired T4 wall-clock median speed ratio: **{final_speed["median_wall_speedup"]:.4f}x**
- All gates passed: **{gates["all_pass"]}**

**The '1.2' in the repository name refers to a previous T4 1.203x result.**
Speed is not guaranteed on other hardware, batch sizes or sequence lengths.
The report covers two narrow benchmark shapes on one T4 and does not prove
generally faster or more accurate behavior. The exported model returns
**option scores only**, not the original auxiliary action logits.

## Inference

\`\`\`python
from modeling_laya_speedup import load_laya_speedup
model, tokenizer = load_laya_speedup(
    "vtava/Laya-1.2-speedup", device="cuda", compile_head=True
)
# Prepare typed-decision tensor inputs using TinyCeNN-LM
# standalone_decision.build_sequence and collate_items.
# with torch.inference_mode(): logits, _ = model(**batch)
\`\`\`

Install the packages in \`requirements.txt\` first. The original
pretrained model remains the deployment reference for use cases requiring
action logits or until broader independent tests validate this variant.
"""
(artifacts/"README.md").write_text(card,encoding="utf-8")
# Verify that reloading serialized head recovers the *selected eager* weights.
restored=MarkerAwareScores(teacher,budget=TARGET_BUDGET).eval()
restored.head=copy.deepcopy(teacher.head).to(device)
restored.head.load_state_dict(load_file(str(head_file)),strict=True)
restored=restored.eval()
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    probe_batch=BENCH_BATCHES[0]
    a,_=chosen_eager(**probe_batch)
    b,_=restored(**probe_batch)
    # If training was selected, chosen_eager already contains best state.
    max_abs_err=float((a.float()-b.float()).abs().max())
assert torch.allclose(a.float(),b.float(),atol=1e-5,rtol=1e-5),max_abs_err
print("Saved model reload parity max abs error:",max_abs_err)
print("Model directory:",artifacts)
# Bundle everything for local archival even if publishing is disabled.
zip_path=OUT/"Laya-1.2-speedup-model-artifacts.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in sorted(artifacts.rglob("*")):
        if f.is_file():
            z.write(f,arcname=f.relative_to(artifacts))
print("Downloadable archive:",zip_path)


In [ ]:
#@title 10. Publish validated model to Hugging Face (requires YOUR write login)
from huggingface_hub import HfApi, notebook_login
from google.colab import files
repo_id=None
if not gates["all_pass"]:
    print("VALIDATION FAILED: not publishing a potentially slower/inaccurate model.")
elif PUBLISH_TO_HF:
    # No tokens are written into exported model files or notebooks.
    notebook_login()
    api=HfApi()
    owner=api.whoami()["name"]
    assert owner=="vtava",(
       f"Connected user @{owner}; expected @vtava. Check Hugging Face login.")
    repo_id=f"{owner}/Laya-1.2-speedup"
    url=f"https://huggingface.co/{repo_id}"
    if api.repo_exists(repo_id,repo_type="model"):
        remote=set(api.list_repo_files(repo_id,repo_type="model"))
        harmless={".gitattributes",".gitignore"}
        if remote-harmless and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError(
                f"Existing model repo {url} has files. To overwrite explicitly "
                "set ALLOW_UPDATE_EXISTING=True; existing data is left unchanged.")
        print("Found existing model repo:",url)
    else:
        created=api.create_repo(repo_id,repo_type="model",
                                private=PRIVATE_REPO,exist_ok=False)
        # RepoUrl subclasses str; it has no .repo_url property.
        print("Created model repo:",str(created))
    api.upload_folder(repo_id=repo_id,repo_type="model",
                      folder_path=str(artifacts),
                      commit_message="Publish validated V4.6 marker-aware head and audit")
    expected_files={p.name for p in artifacts.iterdir() if p.is_file()}
    remote=set(api.list_repo_files(repo_id=repo_id,repo_type="model"))
    assert expected_files.issubset(remote),(
        f"Incomplete upload, missing {expected_files-remote}")
    print("MODEL REPOSITORY VERIFIED:",url)
else:
    print("Publish disabled; model remains in the Colab artifacts directory.")
print("Local archive",zip_path)
files.download(str(zip_path))
